In [10]:
import os
import re
import sys
from pathlib import Path
from typing import List, Dict
from langchain_core.prompts import ChatPromptTemplate
import re
import numpy as np
import pandas as pd
import accelerate
import requests
import rapidfuzz
import json
from langchain_core.output_parsers import StrOutputParser
sys.path.append(str(Path.cwd().parent))  # Add parent directory to sys.path for local imports

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from agent.agents.explainer import explain_rule
from agent.tools.mvel_parser_tool import parse_mvel_branches
from agent.llm import get_llm

from sentence_transformers import SentenceTransformer, util

In [11]:
EXPLAIN_PROMPT = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following extracted rule structure into clear English.\n\n"
            "Requirements:\n"
            "- Start with a 1–2 sentence Summary\n"
            "- Then list Decision logic as bullet points\n"
            "- One bullet per branch, in order\n"
            "- Use 'Otherwise,' for the DEFAULT branch\n"
            "- Use provided context to define business terms if relevant\n\n"
            "Context:\n"
            "{context}\n\n"     
            "Rule extraction (JSON):\n"
            "{extraction_json}"
        )
    ])

# NO PARSING
EXPLAIN_V2 = ChatPromptTemplate.from_messages([
        (
            "system",
            "You write for non-technical stakeholders. "
            "Do not mention code, syntax, or programming terms."
        ),
        (
            "user",
            "Convert the following MVEL text into clear, natural English.\n"
            "Explain what the expression does in plain language.\n"
            "Break down each part of the expression step by step.\n"
            "If there are conditions, operators, or functions, describe what they mean.\n"
            "Write the description in sentences. \n"
            "Provide the final meaning as if explaining to someone with no programming background.\n"
            "MVEL TEXT:\n"
            "{mvel_text}"
        )
    ])



ENGLISH_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n"
        
        "ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n(input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')\nUse grouped conditions with parentheses and logical operators (&&, ||) exactly in this style."
        "{english_text}"
    )
])

from langchain_core.prompts import ChatPromptTemplate


ONE_SHOT_E_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n"
        
        "EXAMPLE:\n"
        "ENGLISH REQUIREMENT:\n"
        "Message must be 'MT', estimate must be 'Actual', and clientId must be null or equal to 'N'.\n\n"
         "MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "NOW CONVERT THE FOLLOWING\nd"
        "ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n(input.message != null && input.message.equalsIgnoreCase('MT')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.clientId == null || input.clientId == 'N')\nUse grouped conditions with parentheses and logical operators (&&, ||) exactly in this style."
        "{english_text}"
    )
])

MULTI_SHOT_E_TO_MVEL = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in writing MVEL rules for business systems. "
        "Generate accurate, production-ready MVEL expressions."
    ),
    (
        "user",
        "Convert the following English requirement into a valid MVEL expression.\n"
        "Translate the business logic precisely.\n"
        "Use proper MVEL syntax.\n"
        "Assume referenced variables already exist unless stated otherwise.\n"
        "Add null checks where appropriate.\n"
        "Return ONLY the MVEL expression.\n"
        "Do not include explanations.\n\n"

        "EXAMPLE:\n\n"

        "FIRST ENGLISH REQUIREMENT:\n"
        "Message must be 'MT', estimate must be 'Actual', and clientId must be null or equal to 'N'.\n\n"

        "FIRST MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"

        "SECOND ENGLISH REQUIREMENT:\n"
        "This expression returns true only if message equals 'CASH' and estimateCode equals 'Actual', "
        "ignoring case and ensuring neither value is null.\n\n"

        "SECOND MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('CASH')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual'))\n\n"

        "THIRD ENGLISH REQUIREMENT:\n"
        "This expression returns true only if message equals 'stoink', estimateCode equals 'Actual', "
        "and sender equals 'STOINK ULTRA PRO MAX', ignoring case and ensuring none of the values are null.\n\n"

        "THIRD MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('stoink')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual')) "
        "&& (input.sender != null && input.sender.equalsIgnoreCase('STOINK ULTRA PRO MAX'))\n\n"

        "NOW CONVERT THE FOLLOWING ENGLISH REQUIREMENT:\n"
        "Format the output as a single boolean MVEL expression similar to this pattern:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n"
        "Use grouped conditions with parentheses and logical operators (&&, ||) exactly in this style.\n\n"

        "ENGLISH REQUIREMENT:\n"
        "{english_text}"
    )
]
)

ONE_SHOT_MVEL_TO_E = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in interpreting MVEL rules used in business systems. "
        "Convert MVEL boolean expressions into clear, concise English requirements."
    ),
    (
        "user",
        "Convert the following MVEL expression into a clear English requirement.\n"
        "Translate the logic precisely.\n"
        "Explain null-check logic explicitly (e.g., 'is null' / 'is not null').\n"
        "Interpret equalsIgnoreCase as 'equals (case-insensitive)'.\n"
        "Preserve AND/OR relationships exactly.\n"
        "Return ONLY the English requirement as a single sentence.\n"
        "Do not include explanations.\n\n"
        
        
         "<instructions>\n"
        "1) Identify each top-level condition group separated by logical operators (&&, ||).\n"
        "2) For each group, translate null checks:\n"
        "   - \"x != null\" -> \"x is not null\"\n"
        "   - \"x == null\" -> \"x is null\"\n"
        "3) Translate string comparisons:\n"
        "   - \"x.equalsIgnoreCase('VALUE')\" -> \"x equals 'VALUE' (case-insensitive)\"\n"
        "   - \"x == 'VALUE'\" -> \"x equals 'VALUE'\"\n"
        "4) Preserve operator meaning and grouping exactly:\n"
        "   - \"&&\" -> \"and\"\n"
        "   - \"||\" -> \"or\"\n"
        "   - Parentheses indicate grouping and must be reflected in the English logic.\n"
        "5) Combine the translated groups into a single sentence that clearly states when the expression returns true.\n"
        "6) Return ONLY the final English requirement sentence. Do not include the steps or any extra text.\n\n"
          "<instructions>\n"
        "EXAMPLES:\n\n"

        
        "EXAMPLE:\n"
        "MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'MT' (case-insensitive) and estimate equals 'Actual' (case-insensitive), and clientId is either null or equals 'N'.\n\n"
        "NOW CONVERT THE FOLLOWING MVEL EXPRESSION:\n"
        "{mvel_text}"
    )
])

MULTI_SHOT_MVEL_TO_E = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an expert in interpreting MVEL rules used in business systems. "
        "Convert MVEL boolean expressions into clear, concise English requirements."
    ),
    (
        "user",
        "Convert the following MVEL expression into a clear English requirement.\n"
        "Translate the logic precisely.\n"
        "Explain null-check logic explicitly (e.g., 'is null' / 'is not null').\n"
        "Interpret equalsIgnoreCase as 'equals (case-insensitive)'.\n"
        "Preserve AND/OR relationships exactly.\n"
        "Return ONLY the English requirement as a single sentence.\n"
        "Do not include explanations.\n\n"
        
         "<instructions>\n"
        "1) Identify each top-level condition group separated by logical operators (&&, ||).\n"
        "2) For each group, translate null checks:\n"
        "   - \"x != null\" -> \"x is not null\"\n"
        "   - \"x == null\" -> \"x is null\"\n"
        "3) Translate string comparisons:\n"
        "   - \"x.equalsIgnoreCase('VALUE')\" -> \"x equals 'VALUE' (case-insensitive)\"\n"
        "   - \"x == 'VALUE'\" -> \"x equals 'VALUE'\"\n"
        "4) Preserve operator meaning and grouping exactly:\n"
        "   - \"&&\" -> \"and\"\n"
        "   - \"||\" -> \"or\"\n"
        "   - Parentheses indicate grouping and must be reflected in the English logic.\n"
        "5) Combine the translated groups into a single sentence that clearly states when the expression returns true.\n"
        "6) Return ONLY the final English requirement sentence. Do not include the steps or any extra text.\n\n"
        "<instructions>\n"
        "EXAMPLES:\n\n"

        
        "EXAMPLES:\n\n"
        "FIRST MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('MT')) "
        "&& (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) "
        "&& (input.clientId == null || input.clientId == 'N')\n\n"
        "FIRST ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'MT' (case-insensitive) and estimate equals 'Actual' (case-insensitive), and clientId is either null or equals 'N'.\n\n"
        "SECOND MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('CASH')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual'))\n\n"
        "SECOND ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'CASH' (case-insensitive) and estimateCode equals 'Actual' (case-insensitive), and neither value is null.\n\n"
        "THIRD MVEL EXPRESSION:\n"
        "(input.message != null && input.message.equalsIgnoreCase('stoink')) "
        "&& (input.estimateCode != null && input.estimateCode.equalsIgnoreCase('Actual')) "
        "&& (input.sender != null && input.sender.equalsIgnoreCase('STOINK ULTRA PRO MAX'))\n\n"
        "THIRD ENGLISH REQUIREMENT:\n"
        "Return true only if message equals 'stoink' (case-insensitive), estimateCode equals 'Actual' (case-insensitive), and sender equals 'STOINK ULTRA PRO MAX' (case-insensitive), and none of those values are null.\n\n"
        "NOW CONVERT THE FOLLOWING MVEL EXPRESSION:\n"
        "{mvel_text}"
    )
])



JUDGE_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an impartial judge evaluating two explanations written for non-technical stakeholders.\n"
        "You must return ONLY valid JSON with no extra text."
    ),
    (
        "user",
        "Evaluate which explanation better satisfies the requirements below.\n\n"
        "Evaluation Criteria (in priority order):\n"
        "1) Clarity for non-technical stakeholders\n"
        "2) Correct interpretation of the rule logic\n"
        "3) Proper structure:\n"
        "   - Starts with a 1–2 sentence Summary\n"
        "   - Then lists Decision logic as bullet points\n"
        "   - One bullet per branch, in order\n"
        "   - Uses 'Otherwise,' for the DEFAULT branch\n"
        "4) Uses provided context appropriately to define business terms\n"
        "5) Does NOT mention code, syntax, JSON, or programming terms\n\n"
        "Rules:\n"
        "- Prefer explanations that are accurate and easy to understand.\n"
        "- Penalize any mention of technical or programming language.\n"
        "- If both are flawed, choose the less flawed one.\n"
        "- Ignore formatting differences unless they affect clarity or correctness.\n"
        "- Do NOT be influenced by length alone.\n\n"
        "Context:\n"
        "{context}\n\n"
        "Rule extraction (JSON):\n"
        "{extraction_json}\n\n"
        "EXPLANATION A:\n"
        "{response_a}\n\n"
        "EXPLANATION B:\n"
        "{response_b}\n\n"
        "Return ONLY valid JSON:\n"
        "{{\n"
        '  "winner": "A" | "B",\n'
        '  "confidence": 0.0-1.0,\n'
        '  "reason": "1-3 sentences explaining the key deciding factors"\n'
        "}}"
    ),
])





In [12]:
# Initialize an LLM and provide well-named helper functions for the notebook
# Keeps functions explicit and easy for users to call.
llm = get_llm(model="llama3.1", temperature=0.0)

def explain_parsed_rule(rule_text: str, context: str = "Business context or glossary here") -> str:
    """Parse MVEL text, run the explain prompt pipeline, and return English.
    Uses the parsed extraction JSON as the prompt input.
    """
    parsed = parse_mvel_branches(rule_text)
    extraction_json = json.dumps(parsed, ensure_ascii=False, indent=2)
    chain = EXPLAIN_PROMPT | llm | StrOutputParser()
    print("[explain_parsed_rule] Running explanation chain on parsed extraction...")
    explanation = chain.invoke({"extraction_json": extraction_json, "context": context}).strip()
    print("[explain_parsed_rule] Completed.")
    return explanation

def explain_raw_mvel(rule_text: str) -> str:
    """Explain raw MVEL text without parsing. Useful when parser fails.
    """
    chain = EXPLAIN_V2 | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content

def explain_raw_mvel_one(rule_text: str) -> str:
    """Explain raw MVEL text without parsing. Useful when parser fails.
    """
    chain = ONE_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content

def explain_raw_mvel_multi(rule_text: str) -> str:
    """Explain raw MVEL text without parsing. Useful when parser fails.
    """
    chain = MULTI_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    print("[explain_raw_mvel] Completed.")
    return response.content

def convert_english_to_mvel(english_text: str) -> str:
    """Convert a plain-English requirement into an MVEL expression via the LLM.
    Returns the raw string returned by the model (intended to be a single MVEL expression).
    """
    chain = ENGLISH_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content


def convert_english_to_mvel_one(english_text: str) -> str:
    """Convert a plain-English requirement into an MVEL expression via the LLM.
    Returns the raw string returned by the model (intended to be a single MVEL expression).
    """
    chain = ONE_SHOT_E_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content

def convert_english_to_mvel_multi(english_text: str) -> str:
    """Convert a plain-English requirement into an MVEL expression via the LLM.
    Returns the raw string returned by the model (intended to be a single MVEL expression).
    """
    chain = MULTI_SHOT_E_TO_MVEL | llm
    print("[convert_english_to_mvel] Converting English to MVEL...")
    response = chain.invoke({"english_text": english_text})
    print("[convert_english_to_mvel] Completed.")
    return response.content


def chain_of_thought(rule_text):
    chain = ONE_SHOT_MVEL_TO_E | llm
    print("[explain_raw_mvel] Running raw-text explanation...")
    response = chain.invoke({"mvel_text": rule_text})
    response.content
    

In [13]:
# Utilities for fuzzy matching and evaluation against a CSV library of rules
import re
import pandas as pd
from rapidfuzz import fuzz

CSV_PATH = "pre_match.csv"

# Example test MVELs (replace or extend as needed)
tests = [
    "(input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')",
    "(input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType.equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') && (input.transactionIndex == 'N') && (input.Ref.contains('CASH MONEY HEROES'))",
    "(input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transactionCode.equalsIgnoreCase('CAPITAL HILL')) && (input.entityFlag != null && input.entityFlag == 'Y')",
]

english_texts = []  # populated by evaluate

def normalize(text: str) -> str:
    t = str(text).strip()
    t = re.sub(r"\s+", " ", t)
    t = t.replace(" | | ", " || ")
    return t

def similarity(a: str, b: str) -> float:
    # Returns 0..1 similarity score using rapidfuzz token_set_ratio
    return fuzz.token_set_ratio(normalize(a), normalize(b)) / 100.0


def cosine_sim(a, b):
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embedding1 = model.encode(a, convert_to_tensor=True)
    embedding2 = model.encode(b, convert_to_tensor=True)
    similarity = util.cos_sim(embedding1, embedding2)
    return similarity.item()


def best_target_match(model_output: str, library_targets: list[str], similarity) -> tuple[int, float]:
    scores = []
    for t in library_targets:
        score = similarity(model_output, t)
        scores.append(score)
    best_idx = max(range(len(scores)), key=lambda i: scores[i])
    print("generated output:", model_output)
    print("matched label:", library_targets[best_idx])
    print(f"{similarity}", float(scores[best_idx]))
    return best_idx, float(scores[best_idx])

def evaluate(explain_fn):
    """Evaluate a given explanation function against the CSV library.
    explain_fn: callable that accepts an MVEL string and returns English text.
    """
    df = pd.read_csv(CSV_PATH)
    df = df.dropna(subset=["RULE", "TARGET"]).reset_index(drop=True)

    library_rules = df["RULE"].astype(str).tolist()
    library_targets = df["TARGET"].astype(str).tolist()

    print(f"Loaded library rows: {len(library_rules)}")

    for i, test_rule in enumerate(tests, start=1):
        print("\n" + "=" * 80)
        print(f"TEST {i}")
        print("TEST RULE:\n", test_rule)

        # Call the provided explain function
        model_output = explain_fn(test_rule)
        english_texts.append(model_output)

        best_idx, best_score = best_target_match(model_output, library_targets, similarity=cosine_sim)


def eval_convert(conversion):
    df = pd.read_csv(CSV_PATH)
    library_rules = df["RULE"].astype(str).tolist()
    explanation = df["TARGET"].astype(str).tolist()

    for idx, rule in enumerate(english_texts):
        # Call conversion and normalize to a single-line MVEL string
        output = conversion(rule)
        output_str = str(output)
        clean_rule = output_str.replace('`', '').replace('\n', ' ').strip()

        # Find best match in the dataset (compare converted MVEL to library rules)
        best_idx, best_score = best_target_match(clean_rule, library_targets=library_rules, similarity=similarity)


def self_consistency_cot(runs: int = 3):
    result = []
    for _ in range(runs):
        result.append(chain_of_thought(tests[0]))
    
    res_concat = "\;".join(result)
    self_consistency_prompt = f"You will get multiple answers in <<>>, seperated by ; <<{res_concat}>> Extract the best answer among those answers."
    self_consistency_prompt_concat = ";".join(self_consistency_prompt)
    messages = [
        ("system", "You are helpful assistance and answer precise and concise"),
        ("user", f"{self_consistency_prompt_concat}")
    ]
    prompt = ChatPromptTemplate.from_messages(messages=messages)
    chain = prompt | llm 
    return chain.content
        

<>:91: SyntaxWarning: invalid escape sequence '\;'
<>:91: SyntaxWarning: invalid escape sequence '\;'
C:\Users\fahud\AppData\Local\Temp\ipykernel_29392\2443713237.py:91: SyntaxWarning: invalid escape sequence '\;'
  res_concat = "\;".join(result)


In [14]:
# Run evaluation using the parsed-rule explainer (preferred when parser works)
evaluate(explain_fn=explain_parsed_rule)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_parsed_rule] Running explanation chain on parsed extraction...
[explain_parsed_rule] Completed.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 732.24it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1030.74it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 972.36it/s, Materializing param=p

generated output: **Summary**

This rule checks if a message is related to a specific transaction and advise, and then applies certain conditions based on the input data.

**Decision Logic**

• If the message is 'MT', 'saipan', or 'CA' (case-insensitive) and the money code is 'Actual' (case-insensitive), and the transaction is 'CAPITAL HILL' (case-insensitive), and advise is 'Y', then proceed to the next step.
 
**Note:** This rule does not have any additional branches, so it will only follow this specific set of conditions.
matched label: If the message is not null and equals "ABC"  where it is case-sensitive, and the moneyCode is not null and equals "Money"  where it is case-sensitive, and the workType is not null and equals "regular"  where it is case-sensitive, and the transaction is not null and equals "Y", then the condition is true.
<function cosine_sim at 0x000001F9390BFCE0> 0.561228334903717

TEST 2
TEST RULE:
 (input.message != null && input.message.equalsIgnoreCase('ABC')) &

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 850.76it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 969.24it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1020.72it/s, Materializing param=p

generated output: **Summary**

This rule applies to specific input conditions and determines whether a transaction is eligible for processing. It checks various criteria, including the message type, estimate type, source work type, client ID, and reference.

**Decision Logic**

* The input message must be 'ABC' (case-insensitive).
* The input estimate must be 'money' (case-insensitive).
* The source work type must be 'normal'.
* The client ID cannot be 'Y'.
* The transaction index must be 'N'.
* The reference must contain the string 'CASH MONEY HEROES'.

**Note**: If none of these conditions are met, the DEFAULT branch is triggered.
matched label: If the message is "MT"  where it is case-sensitive, the moneyCode is not null and the estimateCode is "Money"  where it is case-sensitive, the clientId is either null or "N", and the Cashref is either null or not equal to "TERMINATED", then the condition is true.
<function cosine_sim at 0x000001F9390BFCE0> 0.5528832077980042

TEST 3
TEST RULE

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 994.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1047.49it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 945.72it/s, Materializing param=p

generated output: **Summary**

This rule checks if a specific set of conditions are met for a transaction to be processed. It verifies that the message type is one of four allowed types, the estimate is marked as 'Actual', the transaction code is 'CAPITAL HILL', and an entity flag is set to 'Y'.

**Decision Logic**

* The input message must contain one of the following values: FT900, FT500, SSN, or MT.
* The estimate must be marked as 'Actual'.
* The transaction code must be 'CAPITAL HILL'.
* An entity flag must be set to 'Y'.
matched label: If the message is not null and equals "CASH", where it is case-sensitive and the estimateCode is not null and equals "Actual", where this case-sensitive.
<function cosine_sim at 0x000001F9390BFCE0> 0.4902571141719818


In [15]:
evaluate(explain_fn=explain_raw_mvel)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1002.70it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1025.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1000.37it/s, Materializing param=

generated output: Here's the explanation of the MVEL text in clear, natural English:

**What the expression does:**
This expression checks if a set of conditions are met for an input message. If all conditions are true, it returns true; otherwise, it returns false.

**Breaking down each part step by step:**

1. `input.message != null`: This condition checks if the "message" field in the input data is not empty or null.
2. `(input.message.equalsIgnoreCase('MT') || ... )`: If the message is not null, this condition checks if it matches one of four specific values:
	* 'MT'
	* 'saipan' (note: there's a duplicate 'MT' here, which seems to be an error)
	* 'CA'
	These values are checked using the `equalsIgnoreCase` function, which means it doesn't care about case differences (e.g., 'mt', 'MT', or 'Mt' would all match).
3. `(input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual'))`: This condition checks two things:
	* The "moneyCode" field in the input data is not empty or null.

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 893.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 996.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1033.54it/s, Materializing param=p

generated output: Here's the explanation of the MVEL text in plain language:

**What the expression does:**
This expression checks if a set of conditions are met for an input object. If all conditions are true, it returns true; otherwise, it returns false.

**Breaking down each part step by step:**

1. `input.message != null`: This condition checks if the "message" field in the input object is not empty (i.e., it has a value).
2. `&&` (and operator): If the previous condition is true, this operator ensures that only if both conditions are met will the expression proceed to the next step.
3. `input.message.equalsIgnoreCase('ABC')`: This condition checks if the "message" field in the input object is exactly equal to the string 'ABC', ignoring any case differences (i.e., it's not sensitive to whether the letters are uppercase or lowercase).
4. The same pattern continues for the other conditions:
	* `input.estimate != null && input.estimate.equalsIgnoreCase('money')`: Checks if "estimate" 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1028.43it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1025.66it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1019.97it/s, Materializing param=

generated output: Here's the explanation of the MVEL text in plain language:

**What the expression does:**
This expression checks if a set of conditions are met for an input message. If all conditions are true, then it returns true; otherwise, it returns false.

**Breaking down each part step by step:**

1. `input.message != null`: This condition checks if the "message" field in the input data is not empty or null.
2. `(input.message.equalsIgnoreCase('FT900') || ... )`: This is a group of conditions that check if the "message" field matches certain values. The `equalsIgnoreCase` function compares two strings without considering case (i.e., it's case-insensitive). The values being checked are:
	* 'FT900'
	* 'FT500' (matched by comparing the "messageTypeCode" field)
	* 'SSN'
	* 'MT'
3. `(input.estimate != null && input.estimate.equalsIgnoreCase('Actual'))`: This condition checks if two fields in the input data are not empty or null and match a specific value:
	* The "estimate" field is 

In [16]:
self_consistency_cot(3)

[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Running raw-text explanation...


C:\Users\fahud\AppData\Local\Temp\ipykernel_29392\2443713237.py:91: SyntaxWarning: invalid escape sequence '\;'
  res_concat = "\;".join(result)


TypeError: sequence item 0: expected str instance, NoneType found

In [ ]:
# After you have run `evaluate` to populate `english_texts`,
# you can attempt to convert the generated English back to MVEL and compare.
eval_convert(conversion=convert_english_to_mvel)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("nothing trade")) &&
(input.estimateCode != null && input.estimateCode.equalsIgnoreCase("Actual")) &&
(input.sender != null && input.sender.equalsIgnoreCase("DFS")) &&
(input.entityFlag != null && input.entityFlag.equals("N"))
<function similarity at 0x000002A50ACE6520> 0.6619469026548672
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equals

In [ ]:
eval_convert(conversion=convert_english_to_mvel_one)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction == null || input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("ABC")) &&
(input.moneyCode != null && input.moneyCode.equalsIgnoreCase("Money")) &&
(input.workType != null && input.workType.equalsIgnoreCase("regular")) &&
(input.transaction != null && input.transaction.equals("Y"))
<function similarity at 0x000002A50ACE6520> 0.6678507992895204
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnor

In [ ]:
eval_convert(conversion=convert_english_to_mvel_multi)

[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA')) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise.equalsIgnoreCase('Y'))
matched label: (input.message != null && input.message.equalsIgnoreCase("ABC")) &&
(input.moneyCode != null && input.moneyCode.equalsIgnoreCase("Money")) &&
(input.workType != null && input.workType.equalsIgnoreCase("regular")) &&
(input.transaction != null && input.transaction.equals("Y"))
<function similarity at 0x000002A50ACE6520> 0.6714285714285714
[convert_english_to_mvel] Converting English to MVEL...
[convert_english_to_mvel] Completed.
generated output: (input.message != null && input.message.equalsIgnor

In [ ]:
evaluate(explain_fn=explain_raw_mvel_one)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1003.51it/s, Materializing param=pooler.dense.weight]                            
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 978.78it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 980.01it/s, Materializing param=po

generated output: Return true only if message is not null and equals 'MT' (case-insensitive) or equals 'saipan' (case-insensitive) or equals 'CA' (case-insensitive), and moneyCode is not null and equals 'Actual' (case-insensitive), and transaction is not null and equals 'CAPITAL HILL', and advise is not null and equals 'Y'.
matched label: If the message is not null and equals "ABC"  where it is case-sensitive, and the moneyCode is not null and equals "Money"  where it is case-sensitive, and the workType is not null and equals "regular"  where it is case-sensitive, and the transaction is not null and equals "Y", then the condition is true.
<function cosine_sim at 0x000002A50ACE65C0> 0.7988994121551514

TEST 2
TEST RULE:
 (input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType.equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') &

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 914.17it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 940.51it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 900.39it/s, Materializing param=po

generated output: Return true only if message equals 'ABC' (case-insensitive) and estimate equals 'money' (case-insensitive), and srcworkType is not null and equals 'normal' (case-insensitive), and clientId is not null and does not equal 'Y', and transactionIndex equals 'N', and Ref contains 'CASH MONEY HEROES'.
matched label: If the message is "MT"  where it is case-sensitive, the moneyCode is not null and the estimateCode is "Money"  where it is case-sensitive, the clientId is either null or "N", and the Cashref is either null or not equal to "TERMINATED", then the condition is true.
<function cosine_sim at 0x000002A50ACE65C0> 0.7377310991287231

TEST 3
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 680.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 760.35it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 845.81it/s, Materializing param=po

generated output: Return true only if message is not null and either equals 'FT900' (case-insensitive), equals 'SSN' (case-insensitive), equals 'MT' (case-insensitive), or equals 'FT500' (case-insensitive) and estimate equals 'Actual' (case-insensitive), and transaction is not null and equals 'CAPITAL HILL' (case-insensitive), and entityFlag is not null and equals 'Y'.
matched label: If the message is not null and equals "nothing trade"  where it is case-sensitive, and the estimateCode is not null and equals "Actual"  where it is case-sensitive, and the sender is not null and equals "DFS"  where it is case-sensitive, and the entityFlag is not null and equals "N".
<function cosine_sim at 0x000002A50ACE65C0> 0.7039244771003723


In [ ]:
evaluate(explain_fn=explain_raw_mvel_multi)

Loaded library rows: 7

TEST 1
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('MT') || input.message.equalsIgnoreCase('saipan') || input.message.equalsIgnoreCase('CA'))) && (input.moneyCode != null && input.moneyCode.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transaction.equalsIgnoreCase('CAPITAL HILL')) && (input.advise != null && input.advise == 'Y')
[explain_raw_mvel] Running raw-text explanation...
[explain_raw_mvel] Completed.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 863.44it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 856.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 827.27it/s, Materializing param=po

generated output: Return true only if message equals 'MT' (case-insensitive) or equals 'saipan' (case-insensitive) or equals 'CA' (case-insensitive), and moneyCode equals 'Actual' (case-insensitive), and transaction equals 'CAPITAL HILL' (case-insensitive), and advise equals 'Y', and none of those values are null.
matched label: If the message is not null and equals "ABC"  where it is case-sensitive, and the moneyCode is not null and equals "Money"  where it is case-sensitive, and the workType is not null and equals "regular"  where it is case-sensitive, and the transaction is not null and equals "Y", then the condition is true.
<function cosine_sim at 0x000002A50ACE65C0> 0.7932953834533691

TEST 2
TEST RULE:
 (input.message != null && input.message.equalsIgnoreCase('ABC')) && (input.estimate != null && input.estimate.equalsIgnoreCase('money')) && (input.srcworkType != null && input.srcworkType.equalsIgnoreCase('normal')) && (input.clientId != null && input.clientId != 'Y') && (input.t

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 774.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 765.55it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 901.06it/s, Materializing param=po

generated output: Return true only if message equals 'ABC' (case-insensitive), estimate equals 'money' (case-insensitive), srcworkType equals 'normal' (case-insensitive), clientId is not null and does not equal 'Y', transactionIndex equals 'N', and Ref contains 'CASH MONEY HEROES'.
matched label: If the message is "MT"  where it is case-sensitive, the moneyCode is not null and the estimateCode is "Money"  where it is case-sensitive, the clientId is either null or "N", and the Cashref is either null or not equal to "TERMINATED", then the condition is true.
<function cosine_sim at 0x000002A50ACE65C0> 0.7299190759658813

TEST 3
TEST RULE:
 (input.message != null && (input.message.equalsIgnoreCase('FT900') || input.messageTypeCode.equalsIgnoreCase('FT500') || input.message.equalsIgnoreCase('SSN') || input.message.equalsIgnoreCase('MT'))) && (input.estimate != null && input.estimate.equalsIgnoreCase('Actual')) && (input.transaction != null && input.transactionCode.equalsIgnoreCase('CAPITAL 

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1019.34it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 776.58it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 945.87it/s, Materializing param=p

generated output: Return true only if message is not null and equals 'FT900' (case-insensitive) or equals 'SSN' (case-insensitive) or equals 'MT' (case-insensitive) or equals 'FT500' (case-insensitive), and estimate equals 'Actual' (case-insensitive), and transaction is not null and transactionCode equals 'CAPITAL HILL' (case-insensitive), and entityFlag is not null and equals 'Y'.
matched label: If the message is not null and equals "nothing trade"  where it is case-sensitive, and the estimateCode is not null and equals "Actual"  where it is case-sensitive, and the sender is not null and equals "DFS"  where it is case-sensitive, and the entityFlag is not null and equals "N".
<function cosine_sim at 0x000002A50ACE65C0> 0.7101517915725708


In [ ]:
# step1_make_prompts.py
import json

IN_PATH = "pre_match_sft_chat.jsonl"
OUT_PATH = "prompts.jsonl"

def build_prompt(messages):
    prompt = ""
    for m in messages:
        if m["role"] in ["system", "user"]:
            prompt += f"{m['role'].upper()}:\n{m['content']}\n\n"
    prompt += "ASSISTANT:\n"
    return prompt

with open(IN_PATH, "r", encoding="utf-8") as f_in, \
     open(OUT_PATH, "w", encoding="utf-8") as f_out:

    for line in f_in:
        obj = json.loads(line)
        prompt = build_prompt(obj["messages"])
        f_out.write(json.dumps({"prompt": prompt}) + "\n")

print("prompts.jsonl created")


prompts.jsonl created


In [ ]:
# step2_generate_candidates.py
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import accelerate

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # change if needed

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to("cpu")


with open("prompts.jsonl", "r") as f_in, \
     open("candidates.jsonl", "w") as f_out:

    for line in f_in:
        row = json.loads(line)
        prompt = row["prompt"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            num_return_sequences=4,
        )

        candidates = [
            tokenizer.decode(o, skip_special_tokens=True).split("ASSISTANT:")[-1].strip()
            for o in outputs
        ]

        f_out.write(json.dumps({
            "prompt": prompt,
            "candidates": candidates
        }) + "\n")

print("candidates.jsonl created")


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 753.78it/s, Materializing param=model.norm.weight]                              


candidates.jsonl created


In [ ]:
import json
import random

# Reuse the llm initialized earlier in the notebook (via get_llm).
# If llm isn't defined yet, create it using get_llm.
try:
    llm  # existing llm from previous cells
except NameError:
    llm = get_llm(model="phi3:3.8b", temperature=0.0)

judge_chain = JUDGE_PROMPT | llm


def judge(a, b, prompt_blob):
    result = judge_chain.invoke({
        "context": "",                  # keep empty
        "extraction_json": "",          # keep empty
        "response_a": a,
        "response_b": b,
    })
    return json.loads(result.content)["winner"]


def anchor_pick(candidates, prompt_blob):
    wins = [0] * 4
    anchor = random.randrange(4)

    for i in range(4):
        if i == anchor:
            continue

        # flip A/B randomly to reduce bias
        if random.random() < 0.5:
            winner = judge(candidates[anchor], candidates[i], prompt_blob)
            if winner == "A":
                wins[anchor] += 1
            else:
                wins[i] += 1
        else:
            winner = judge(candidates[i], candidates[anchor], prompt_blob)
            if winner == "A":
                wins[i] += 1
            else:
                wins[anchor] += 1

    best = wins.index(max(wins))
    worst = wins.index(min(wins))
    return best, worst


with open("candidates.jsonl") as f_in, open("dpo_pairs.jsonl", "w") as f_out:
    for line in f_in:
        row = json.loads(line)

        best, worst = anchor_pick(row["candidates"], row["prompt"])

        f_out.write(json.dumps({
            "prompt": row["prompt"],
            "chosen": row["candidates"][best],
            "rejected": row["candidates"][worst],
        }) + "\n")


JSONDecodeError: Expecting ',' delimiter: line 4 column 507 (char 546)

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="dpo_pairs.jsonl")["train"]


Generating train split: 14 examples [00:00, 157.74 examples/s]


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DPOTrainer
from peft import LoraConfig
import torch

model_name = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)

dpo_trainer = DPOTrainer(
    model,
    ref_model=None,  # automatically clones reference
    args=None,
    beta=0.1,
    train_dataset=dataset,
    tokenizer=tokenizer,
    peft_config=peft_config,
)

dpo_trainer.train()

dpo_trainer.save_model("dpo_model")


OSError: The paging file is too small for this operation to complete. (os error 1455)